# Ride Demand Prediction & Driver Allocation System

This notebook implements an end-to-end Machine Learning pipeline for ride demand prediction and driver allocation optimization.

## Steps:
1. **Data Generation**: Create synthetic ride data with realistic patterns.
2. **Preprocessing**: Convert timestamps, aggregate demand by zone, and compute lag features.
3. **Model Training**: Train XGBoost and Random Forest models to forecast demand.
4. **Evaluation**: Compare models using RMSE, MAE, and R² metrics.
5. **Driver Allocation**: Optimize fleet distribution based on predicted demand.
6. **Visualizations**: Interactive heatmaps of demand and allocation strategies.

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from IPython.display import display, IFrame

# Add src to path
sys.path.append(os.path.abspath('../src'))

from data_generation import generate_synthetic_data
from data_processing import process_data
from geospatial import create_zone_geodataframe, visualize_demand_heatmap, visualize_zone_clusters
from model import RideDemandPredictor
from driver_allocation import DriverAllocator

# Ensure output directories exist
if not os.path.exists('../data'):
    os.makedirs('../data')
if not os.path.exists('../models'):
    os.makedirs('../models')

ModuleNotFoundError: No module named 'src'

## 1. Data Generation & Processing
We generate synthetic ride data simulating 3 months of activity in a fictional city.

In [ ]:
# Generate Data
generate_synthetic_data(num_rides=50000)

# Process & Engineer Features
process_data(input_file='../data/raw_rides.csv', output_file='../data/processed_demand.csv')

## 2. Load & Explore Data

In [ ]:
df = pd.read_csv('../data/processed_demand.csv')
print(f"Dataset Shape: {df.shape}")
display(df.head())

## 3. Model Training & Evaluation
We train XGBoost and Random Forest regressors to predict demand.

In [ ]:
predictor = RideDemandPredictor()
X_train, y_train, X_test, y_test = predictor.split_data(df)

print(f"Training Set: {X_train.shape}, Test Set: {X_test.shape}")

# Train Models
models = predictor.train_models(X_train, y_train)

# Evaluate
metrics = predictor.evaluate(models, X_test, y_test)

pd.DataFrame(metrics).T

## 4. Driver Allocation Strategy
Using the trained model, we forecast demand and allocate drivers to minimize unmet demand.

In [ ]:
best_model = models['XGBoost']
allocator = DriverAllocator(best_model)

# Simulate allocation for a sample of zones
sample_features = X_test.sample(50).copy()
sample_predictions = allocator.predict_demand(sample_features)

allocation_df = sample_features.copy()
allocation_df['predicted_demand'] = sample_predictions
allocation_df['zone_id'] = df.loc[sample_features.index, 'zone_id']

# Optimize for 100 available drivers
allocation_results = allocator.optimize_allocation(allocation_df, total_drivers=100)

display(allocation_results[['zone_id', 'predicted_demand', 'allocated_drivers', 'surge_prob']].head(10))

## 5. Visualizations
Analyze spatial demand patterns.

In [2]:
# Generate Heatmap using Raw Data
raw_df = pd.read_csv('../data/raw_rides.csv')
visualize_demand_heatmap(raw_df.sample(2000), output_map='../notebooks/demand_heatmap.html')

print("Heatmap saved to notebooks/demand_heatmap.html")


NameError: name 'visualize_demand_heatmap' is not defined